# Lightweight Heterogeneity-Aware Federated Web Crawling Detection

This notebook implements a reproducible, configurable pipeline:
- CommonCrawl WARC metadata ingestion (optional) and local access-log ingestion
- Session/window feature extraction and time/site-aware splits
- Lightweight temporal + behavioural detector
- Simulated multi-site clients, FedProx local updates, reliability-weighted aggregation, and local adapters
- Calibration, entropy/PSI drift, retrieval-assisted incident summaries, policy-based mitigation
- Confusion matrix, ROC, accuracy/loss curves, metric comparison, ablation, and resource measurements

**Important:** Common Crawl is a web archive, not a labeled web-server access-log dataset. For training, provide locally collected access logs or a compatible labeled CSV. The notebook does not fabricate measured results; it evaluates on the data you supply.

## 1. Setup

In [ ]:
# Run in Colab or a local Jupyter environment
!pip -q install pandas numpy scikit-learn matplotlib seaborn tensorflow requests warcio rank-bm25 sentence-transformers

In [ ]:
import os, re, json, time, math, random, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

DATA_PATH = "/content/web_traffic.csv"   # change to your labeled access-log CSV
SAVE_DIR = "Finance 4"                  # current project output directory
os.makedirs(SAVE_DIR, exist_ok=True)

LABEL_COL = "label"
SITE_COL = "site_id"
TIME_COL = "timestamp"
SESSION_COL = "session_id"
URL_COL = "url"
UA_COL = "user_agent"

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load and normalize access logs

In [ ]:
def load_traffic(path=DATA_PATH):
    df = pd.read_csv(path)
    required = [TIME_COL, URL_COL, LABEL_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. CSV must include {required}.")
    defaults = {
        SITE_COL: "site_0", SESSION_COL: None, "method": "GET",
        "status_code": 200, "response_size": 0, UA_COL: "unknown"
    }
    for col, val in defaults.items():
        if col not in df:
            df[col] = val
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL, URL_COL, LABEL_COL]).copy()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df["method"] = df["method"].astype(str).str.upper()
    df["status_code"] = pd.to_numeric(df["status_code"], errors="coerce").fillna(0).astype(int)
    df["response_size"] = pd.to_numeric(df["response_size"], errors="coerce").fillna(0)
    df[URL_COL] = df[URL_COL].astype(str)
    df[UA_COL] = df[UA_COL].fillna("unknown").astype(str)
    return df.sort_values([SITE_COL, TIME_COL]).reset_index(drop=True)

# Uncomment after placing your CSV:
# traffic = load_traffic()
# traffic.head()

## 3. Sessionization and behavioural feature extraction

In [ ]:
def path_tokens(url):
    try:
        from urllib.parse import urlparse
        p = urlparse(url if "://" in url else "https://local"+url).path
    except Exception:
        p = str(url).split("?")[0]
    return [x for x in p.lower().split("/") if x]

def add_sessions(df, gap_minutes=30):
    df = df.sort_values([SITE_COL, TIME_COL]).copy()
    if df[SESSION_COL].notna().any():
        df[SESSION_COL] = df[SESSION_COL].fillna("missing")
        return df
    delta = df.groupby(SITE_COL)[TIME_COL].diff().dt.total_seconds().div(60)
    new_session = delta.isna() | (delta > gap_minutes)
    df[SESSION_COL] = df[SITE_COL].astype(str) + "_" + new_session.groupby(df[SITE_COL]).cumsum().astype(str)
    return df

def make_session_features(df, window=10):
    df = add_sessions(df)
    rows = []
    for (site, sid), g in df.groupby([SITE_COL, SESSION_COL], sort=False):
        g = g.sort_values(TIME_COL)
        ts = g[TIME_COL].astype("int64").to_numpy() / 1e9
        gaps = np.diff(ts)
        paths = [path_tokens(u) for u in g[URL_COL]]
        flat_paths = ["/".join(p) for p in paths]
        status = g["status_code"].to_numpy()
        methods = g["method"].to_numpy()
        ua = g[UA_COL].astype(str)
        dur = max(float(ts[-1]-ts[0]), 0.0) if len(ts)>1 else 0.0
        n = len(g)
        row = {
            SITE_COL: site, SESSION_COL: sid, TIME_COL: g[TIME_COL].iloc[0],
            LABEL_COL: g[LABEL_COL].mode().iloc[0],
            "request_count": n, "request_rate": n/max(dur,1.0),
            "iat_mean": float(np.mean(gaps)) if len(gaps) else 0.0,
            "iat_std": float(np.std(gaps)) if len(gaps) else 0.0,
            "iat_cv": float(np.std(gaps)/(np.mean(gaps)+1e-6)) if len(gaps) else 0.0,
            "url_unique_ratio": len(set(flat_paths))/max(n,1),
            "url_repeat_ratio": 1-len(set(flat_paths))/max(n,1),
            "path_depth_mean": float(np.mean([len(p) for p in paths])),
            "path_token_diversity": len(set(t for p in paths for t in p))/max(sum(map(len,paths)),1),
            "method_get_ratio": float(np.mean(methods=="GET")),
            "method_post_ratio": float(np.mean(methods=="POST")),
            "status_2xx_ratio": float(np.mean((status>=200)&(status<300))),
            "status_4xx_ratio": float(np.mean((status>=400)&(status<500))),
            "status_5xx_ratio": float(np.mean(status>=500)),
            "response_size_mean": float(g["response_size"].mean()),
            "response_size_std": float(g["response_size"].std(ddof=0)),
            "ua_unique_ratio": ua.nunique()/max(n,1),
            "session_duration": dur
        }
        rows.append(row)
    return pd.DataFrame(rows)

# features = make_session_features(traffic)
# features.head()

## 4. Labels, temporal split, and robust scaling

In [ ]:
# Map labels consistently; edit aliases to match your dataset.
LABEL_ALIASES = {
    "benign": "benign", "legitimate": "benign", "normal": "benign",
    "crawler": "benign_crawler", "legitimate_crawler": "benign_crawler",
    "aggressive": "aggressive_crawler", "aggressive_crawling": "aggressive_crawler",
    "suspicious": "suspicious_automation", "bot": "suspicious_automation",
    "malicious": "malicious", "attack": "malicious"
}
def normalize_labels(s):
    s = str(s).strip().lower().replace(" ", "_")
    return LABEL_ALIASES.get(s, s)

def temporal_site_split(df, test_frac=0.20, val_frac=0.15):
    df = df.sort_values([SITE_COL, TIME_COL]).copy()
    train_parts, val_parts, test_parts = [], [], []
    for _, g in df.groupby(SITE_COL):
        n=len(g); n_test=max(1,int(n*test_frac)); n_val=max(1,int(n*val_frac))
        test_parts.append(g.iloc[-n_test:])
        val_parts.append(g.iloc[-(n_test+n_val):-n_test])
        train_parts.append(g.iloc[:-(n_test+n_val)])
    return (pd.concat(train_parts).reset_index(drop=True),
            pd.concat(val_parts).reset_index(drop=True),
            pd.concat(test_parts).reset_index(drop=True))

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

FEATURE_COLS = [
    "request_count","request_rate","iat_mean","iat_std","iat_cv",
    "url_unique_ratio","url_repeat_ratio","path_depth_mean","path_token_diversity",
    "method_get_ratio","method_post_ratio","status_2xx_ratio","status_4xx_ratio",
    "status_5xx_ratio","response_size_mean","response_size_std",
    "ua_unique_ratio","session_duration"
]

## 5. Sequence construction

In [ ]:
def make_sequences(frame, feature_cols, time_steps=10):
    X, y, sites, times = [], [], [], []
    for _, g in frame.groupby(SITE_COL, sort=False):
        g=g.sort_values(TIME_COL)
        vals=g[feature_cols].to_numpy(dtype=np.float32)
        labels=g["label_id"].to_numpy()
        for end in range(time_steps, len(g)+1):
            X.append(vals[end-time_steps:end])
            # sequence label: most recent session label
            y.append(labels[end-1]); sites.append(g[SITE_COL].iloc[end-1]); times.append(g[TIME_COL].iloc[end-1])
    return np.asarray(X,np.float32), np.asarray(y,np.int32), np.asarray(sites), np.asarray(times)

# Full data preparation after loading:
# traffic[LABEL_COL] = traffic[LABEL_COL].map(normalize_labels)
# features = make_session_features(traffic)
# train_df, val_df, test_df = temporal_site_split(features)
# le = LabelEncoder()
# train_df["label_id"] = le.fit_transform(train_df[LABEL_COL])
# val_df["label_id"] = le.transform(val_df[LABEL_COL])
# test_df["label_id"] = le.transform(test_df[LABEL_COL])
# scaler = RobustScaler()
# train_df[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
# val_df[FEATURE_COLS] = scaler.transform(val_df[FEATURE_COLS])
# test_df[FEATURE_COLS] = scaler.transform(test_df[FEATURE_COLS])
# X_train,y_train,site_train,t_train = make_sequences(train_df,FEATURE_COLS)
# X_val,y_val,site_val,t_val = make_sequences(val_df,FEATURE_COLS)
# X_test,y_test,site_test,t_test = make_sequences(test_df,FEATURE_COLS)

## 6. Lightweight hybrid detector

In [ ]:
def build_detector(time_steps, n_features, n_classes, width=64, dropout=0.20):
    inp=tf.keras.Input((time_steps,n_features), name="traffic_sequence")
    branches=[]
    for k in (3,5,7):
        x=tf.keras.layers.SeparableConv1D(width, k, padding="same", activation="swish")(inp)
        x=tf.keras.layers.BatchNormalization()(x)
        x=tf.keras.layers.GlobalAveragePooling1D()(x)
        branches.append(x)
    temporal=tf.keras.layers.Concatenate()(branches)
    temporal=tf.keras.layers.Dense(128,activation="swish")(temporal)
    temporal=tf.keras.layers.Dropout(dropout)(temporal)

    b=tf.keras.layers.Dense(128,activation="swish")(inp[:,-1,:])
    gate=tf.keras.layers.Dense(128,activation="sigmoid")(tf.keras.layers.Concatenate()([temporal,b]))
    fused=gate*temporal+(1-gate)*b
    fused=tf.keras.layers.Dense(96,activation="swish")(fused)
    fused=tf.keras.layers.Dropout(dropout)(fused)
    out=tf.keras.layers.Dense(n_classes,activation="softmax",name="traffic_class")(fused)
    return tf.keras.Model(inp,out,name="LightweightHybridDetector")

# model = build_detector(X_train.shape[1], X_train.shape[2], len(le.classes_))
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
#              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
# class_weights = dict(enumerate(weights))
# history = model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=30,
#                    batch_size=256,class_weight=class_weights,
#                    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True),
#                              tf.keras.callbacks.ReduceLROnPlateau(patience=2)])

## 7. Federated clients: FedProx + reliability aggregation + local adapters

In [ ]:
def build_local_model(global_weights, input_shape, n_classes):
    m=build_detector(input_shape[0],input_shape[1],n_classes)
    m.set_weights(global_weights)
    return m

def client_train(client_df, global_weights, input_shape, n_classes, rounds=1,
                 lr=1e-3, mu=1e-2, batch_size=128):
    X,y,_,_=make_sequences(client_df,FEATURE_COLS)
    model=build_local_model(global_weights,input_shape,n_classes)
    optimizer=tf.keras.optimizers.Adam(lr)
    loss_fn=tf.keras.losses.SparseCategoricalCrossentropy()
    global_vars=[tf.convert_to_tensor(w) for w in global_weights]
    for _ in range(rounds):
        order=np.random.permutation(len(X))
        for start in range(0,len(order),batch_size):
            ids=order[start:start+batch_size]
            xb=tf.convert_to_tensor(X[ids]); yb=tf.convert_to_tensor(y[ids])
            with tf.GradientTape() as tape:
                pred=model(xb,training=True)
                loss=loss_fn(yb,pred)
                prox=tf.add_n([tf.reduce_sum(tf.square(v-g)) for v,g in zip(model.trainable_variables,global_vars)])
                total=loss+(mu/2.0)*prox
            grads=tape.gradient(total,model.trainable_variables)
            optimizer.apply_gradients(zip(grads,model.trainable_variables))
    return model.get_weights(), len(X)

def client_reliability(model, val_df, n_classes):
    X,y,_,_=make_sequences(val_df,FEATURE_COLS)
    if len(y)==0: return 0.05
    p=model.predict(X,verbose=0)
    from sklearn.metrics import f1_score
    score=f1_score(y,np.argmax(p,axis=1),average="macro",zero_division=0)
    return float(np.clip(score,0.05,1.0))

def weighted_federated_round(global_model, clients, client_val, input_shape, n_classes,
                             local_epochs=1, mu=1e-2):
    base=global_model.get_weights(); updates=[]; weights=[]
    for site, cdf in clients.items():
        local_w,n=client_train(cdf,base,input_shape,n_classes,rounds=local_epochs,mu=mu)
        local_model=build_local_model(local_w,input_shape,n_classes)
        rel=client_reliability(local_model,client_val.get(site,cdf.iloc[:0]),n_classes)
        updates.append(local_w); weights.append(max(n,1)*rel)
    weights=np.asarray(weights,dtype=float); weights/=weights.sum()+1e-12
    new=[]
    for layer in range(len(base)):
        new.append(np.sum([weights[i]*updates[i][layer] for i in range(len(updates))],axis=0))
    global_model.set_weights(new)
    return global_model, weights

# Split each training site locally. Example:
# client_train_frames={s:g for s,g in train_df.groupby(SITE_COL)}
# client_val_frames={s:g for s,g in val_df.groupby(SITE_COL)}
# global_model=build_detector(X_train.shape[1],X_train.shape[2],len(le.classes_))
# global_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
# for rnd in range(5):
#     global_model, aggregation_weights=weighted_federated_round(
#         global_model,client_train_frames,client_val_frames,
#         X_train.shape[1:],len(le.classes_),local_epochs=1)

## 8. Local personalization adapter

In [ ]:
def build_personalized_adapter(global_model, n_classes, adapter_units=32):
    inp=tf.keras.Input(global_model.input_shape[1:])
    h=global_model.get_layer("traffic_sequence")(inp)
    # Reuse the global representation; small client-specific residual adapter.
    x=tf.keras.layers.SeparableConv1D(64,3,padding="same",activation="swish")(h)
    x=tf.keras.layers.GlobalAveragePooling1D()(x)
    x=tf.keras.layers.Dense(adapter_units,activation="swish")(x)
    x=tf.keras.layers.Dense(n_classes,activation="softmax")(x)
    return tf.keras.Model(inp,x,name="LocalPersonalizedAdapter")

# This adapter is intentionally trained locally and is never sent to the coordinator.
# adapter = build_personalized_adapter(global_model, len(le.classes_))

## 9. Calibration, entropy, and PSI drift

In [ ]:
from sklearn.metrics import log_loss
def predictive_entropy(prob):
    p=np.clip(prob,1e-9,1.0)
    return -np.sum(p*np.log(p),axis=1)

def fit_temperature(logits, y, candidates=np.linspace(0.5,3.0,51)):
    y=np.asarray(y)
    best_t=1.0; best=float("inf")
    for t in candidates:
        z=logits/t; z=z-z.max(axis=1,keepdims=True)
        p=np.exp(z); p/=p.sum(axis=1,keepdims=True)
        score=log_loss(y,p,labels=np.arange(p.shape[1]))
        if score<best: best,best_t=score,float(t)
    return best_t

def apply_temperature(prob, temperature):
    logits=np.log(np.clip(prob,1e-9,1.0))/temperature
    logits-=logits.max(axis=1,keepdims=True)
    p=np.exp(logits); return p/p.sum(axis=1,keepdims=True)

def psi(expected, actual, bins=10):
    expected=np.asarray(expected); actual=np.asarray(actual)
    edges=np.unique(np.quantile(expected,np.linspace(0,1,bins+1)))
    if len(edges)<2: return 0.0
    edges[0]-=1e-9; edges[-1]+=1e-9
    e=np.histogram(expected,bins=edges)[0]/max(len(expected),1)
    a=np.histogram(actual,bins=edges)[0]/max(len(actual),1)
    e=np.clip(e,1e-6,None); a=np.clip(a,1e-6,None)
    return float(np.sum((a-e)*np.log(a/e)))

# logits_val=model.predict(X_val); temp=fit_temperature(np.log(np.clip(logits_val,1e-9,1)),y_val)
# calibrated=apply_temperature(logits_val,temp)
# entropy=predictive_entropy(calibrated)
# drift_scores={f:psi(train_df[f],test_df[f]) for f in FEATURE_COLS}

## 10. Retrieval-augmented incident assistance

In [ ]:
SECURITY_KB = [
 {"id":"KB-01","title":"Verified crawler policy","text":"Verify crawler identity using published IP ranges and reverse/forward DNS where operationally available. Do not rely on user-agent alone. Permit verified crawlers within crawl-delay and site policy."},
 {"id":"KB-02","title":"Rate limiting","text":"Apply token-bucket rate limiting by client, session, and endpoint. Use gradual limits and monitor false positives before broad blocking."},
 {"id":"KB-03","title":"Suspicious traversal","text":"Repeated high-rate traversal, low inter-arrival variability, repeated paths, and unusual response-code sequences may indicate automation. Corroborate multiple signals."},
 {"id":"KB-04","title":"Temporary challenge","text":"For uncertain automation, use a temporary challenge or step-up verification. Avoid challenging known legitimate crawlers or accessibility-sensitive flows."},
 {"id":"KB-05","title":"Incident containment","text":"Block only when policy and corroborating evidence support malicious activity. Log evidence, action, scope, expiry, and rollback identifier."}
]

def retrieve_incident(query, top_k=3):
    try:
        from rank_bm25 import BM25Okapi
        docs=[(x["title"]+" "+x["text"]).lower().split() for x in SECURITY_KB]
        bm=BM25Okapi(docs)
        scores=bm.get_scores(str(query).lower().split())
        idx=np.argsort(scores)[::-1][:top_k]
        return [SECURITY_KB[i] for i in idx if scores[i]>0]
    except Exception:
        terms=set(str(query).lower().split())
        ranked=sorted(SECURITY_KB,key=lambda d:len(terms & set((d["title"]+" "+d["text"]).lower().split())),reverse=True)
        return ranked[:top_k]

def incident_summary(features, risk, uncertainty, kb):
    evidence=", ".join(f"{k}={features.get(k,0):.3g}" for k in
                      ["request_rate","iat_cv","url_repeat_ratio","status_4xx_ratio","ua_unique_ratio"])
    refs="; ".join(x["id"]+": "+x["text"] for x in kb)
    return (f"Risk={risk:.3f}; predictive entropy={uncertainty:.3f}. "
            f"Observed evidence: {evidence}. Retrieved guidance: {refs}")

# For suspicious/uncertain predictions only:
# incident_kb=retrieve_incident("high request rate repeated URL traversal status 4xx")
# print(incident_summary(test_df.iloc[0].to_dict(),float(calibrated[0].max()),float(entropy[0]),incident_kb))

## 11. Policy-constrained adaptive mitigation

In [ ]:
def choose_action(risk, entropy, verified_crawler=False, policy=None):
    policy=policy or {"allow_max_risk":0.25,"challenge_max_risk":0.60,
                       "uncertainty_threshold":0.85}
    if verified_crawler:
        return {"action":"allow","reason":"verified crawler allowlist"}
    if risk < policy["allow_max_risk"]:
        return {"action":"allow","reason":"low calibrated risk"}
    if risk < policy["challenge_max_risk"] or entropy>policy["uncertainty_threshold"]:
        return {"action":"rate_limit_or_challenge","reason":"moderate risk or uncertainty"}
    return {"action":"temporary_block","reason":"high risk; policy review and expiry required"}

class TokenBucket:
    def __init__(self, capacity=20, refill_rate=2):
        self.capacity=float(capacity); self.tokens=float(capacity)
        self.refill_rate=float(refill_rate); self.last=time.time()
    def allow(self, cost=1):
        now=time.time(); self.tokens=min(self.capacity,self.tokens+(now-self.last)*self.refill_rate); self.last=now
        if self.tokens>=cost:
            self.tokens-=cost; return True
        return False

# Always audit and support rollback; this notebook returns decisions but does not block real traffic.
def mitigation_audit(event_id, action, evidence, policy_version="v1"):
    return {"event_id":str(event_id),"action":action["action"],"reason":action["reason"],
            "evidence":evidence,"policy_version":policy_version,"timestamp_utc":pd.Timestamp.utcnow().isoformat(),
            "rollback_required":action["action"]=="temporary_block"}

## 12. Evaluation and plots

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score)

def evaluate_model(model, X, y, class_names, prefix="test"):
    prob=model.predict(X,verbose=0)
    pred=prob.argmax(axis=1)
    acc=accuracy_score(y,pred)
    p,r,f,_=precision_recall_fscore_support(y,pred,average="macro",zero_division=0)
    try: auc_macro=roc_auc_score(y,prob,multi_class="ovr",average="macro")
    except Exception: auc_macro=np.nan
    metrics={"Accuracy":acc,"Precision_macro":p,"Recall_macro":r,"F1_macro":f,"AUC_macro_ovr":auc_macro}
    print(prefix,metrics)
    print(classification_report(y,pred,target_names=class_names,zero_division=0))
    cm=confusion_matrix(y,pred)
    fig,ax=plt.subplots(figsize=(8,7))
    im=ax.imshow(cm,cmap="Blues")
    ax.set_xticks(range(len(class_names)),class_names,rotation=45,ha="right")
    ax.set_yticks(range(len(class_names)),class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{prefix} Confusion Matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    fig.colorbar(im,ax=ax); fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR,f"{prefix}_confusion_matrix.png"),dpi=300,bbox_inches="tight")
    plt.show()

    y_bin=tf.keras.utils.to_categorical(y,num_classes=len(class_names))
    fig,ax=plt.subplots(figsize=(8,7))
    for i,name in enumerate(class_names):
        fpr,tpr,_=roc_curve(y_bin[:,i],prob[:,i])
        ax.plot(fpr,tpr,label=f"{name} (AUC={auc(fpr,tpr):.3f})")
    ax.plot([0,1],[0,1],"k--",linewidth=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{prefix} ROC"); ax.legend(fontsize=8); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR,f"{prefix}_roc.png"),dpi=300,bbox_inches="tight")
    plt.show()
    return metrics,prob,pred

# test_metrics, test_prob, test_pred = evaluate_model(global_model,X_test,y_test,le.classes_)

## 13. Accuracy/loss curves

In [ ]:
def plot_training_history(history, save_name="training_curves.png"):
    h=history.history
    fig,axs=plt.subplots(1,2,figsize=(12,4.5))
    axs[0].plot(h.get("accuracy",[]),label="Train")
    axs[0].plot(h.get("val_accuracy",[]),label="Validation")
    axs[0].set_title("Accuracy"); axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Accuracy"); axs[0].legend(); axs[0].grid(alpha=.25)
    axs[1].plot(h.get("loss",[]),label="Train")
    axs[1].plot(h.get("val_loss",[]),label="Validation")
    axs[1].set_title("Loss"); axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Loss"); axs[1].legend(); axs[1].grid(alpha=.25)
    fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR,save_name),dpi=300,bbox_inches="tight"); plt.show()

# plot_training_history(history)

## 14. Existing-method comparison and ablation

In [ ]:
# Populate these arrays only from actual runs or published values you have verified.
# Leave NaN for methods/metrics not evaluated; do not present invented values as experimental.
comparison = pd.DataFrame({
    "Method":["ObTA","FedPhishLLM","AMD","QFedPhish","Proposed"],
    "Accuracy":[np.nan]*5, "Precision":[np.nan]*5, "Recall":[np.nan]*5,
    "F1":[np.nan]*5, "AUC":[np.nan]*5, "Latency_ms":[np.nan]*5
})
ablation = pd.DataFrame({
    "Variant":["Full model","No multi-resolution temporal branch","No gated interaction",
               "FedAvg instead of reliability aggregation","No local adapter","No calibration/RAG"],
    "Accuracy":[np.nan]*6,"F1":[np.nan]*6,"AUC":[np.nan]*6,
    "Latency_ms":[np.nan]*6
})

def plot_metric_comparison(df, metrics=("Accuracy","F1","AUC"), filename="method_comparison.png"):
    d=df.set_index("Method")[list(metrics)].astype(float)
    ax=d.plot(kind="bar",figsize=(10,5),width=.78)
    ax.set_ylim(0,1); ax.set_ylabel("Score"); ax.set_title("Method Comparison")
    ax.grid(axis="y",alpha=.25); plt.xticks(rotation=20,ha="right"); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,filename),dpi=300,bbox_inches="tight"); plt.show()

# plot_metric_comparison(comparison)

## 15. Communication, latency, and memory

In [ ]:
def model_resource_report(model, X_sample, rounds=1):
    t0=time.perf_counter()
    _=model(X_sample[:min(32,len(X_sample))],training=False)
    latency_ms=(time.perf_counter()-t0)*1000/max(min(32,len(X_sample)),1)
    params=model.count_params()
    weight_bytes=sum(w.nbytes for w in model.get_weights())
    return {"parameters":int(params),"weight_size_MB":weight_bytes/1e6,
            "batch_inference_ms_per_sample":latency_ms}

def communication_report(client_weights, global_weights):
    sent=sum(np.asarray(w).nbytes for w in client_weights)
    total=sum(np.asarray(w).nbytes for w in global_weights)
    return {"client_upload_MB":sent/1e6,"global_model_MB":total/1e6,
            "clients":len(client_weights),"round_upload_MB":sent/1e6}

# print(model_resource_report(global_model,X_test))

## 16. Export metrics and audit artifacts

In [ ]:
def export_results(metrics, comparison_df=None, ablation_df=None, audit_records=None):
    metrics_df=pd.DataFrame([metrics])
    metrics_df.to_excel(os.path.join(SAVE_DIR,"evaluation_metrics.xlsx"),index=False)
    if comparison_df is not None:
        comparison_df.to_excel(os.path.join(SAVE_DIR,"method_comparison.xlsx"),index=False)
    if ablation_df is not None:
        ablation_df.to_excel(os.path.join(SAVE_DIR,"ablation_results.xlsx"),index=False)
    if audit_records is not None:
        pd.DataFrame(audit_records).to_json(os.path.join(SAVE_DIR,"mitigation_audit.json"),orient="records",indent=2)
    print("Saved to:",os.path.abspath(SAVE_DIR))

# export_results(test_metrics, comparison, ablation, audit_records=[])

## Dataset schema

Required CSV columns:
- `timestamp`: parseable timestamp
- `url`: request URL/path
- `label`: locally validated class label

Optional columns:
- `site_id`, `session_id`, `method`, `status_code`, `response_size`, `user_agent`

Recommended labels: `benign`, `benign_crawler`, `aggressive_crawler`, `suspicious_automation`, `malicious`.

Keep raw logs on each site. Only model updates, compact diagnostics, and approved incident summaries should leave the site. For deployment, add authenticated client communication, secure aggregation, update clipping, DP if required by the threat model, and policy review before automated blocking.